## Secrets — like ConfigMaps but for sensitive data

A **Secret** is a ConfigMap plus the handling that sensitive data needs:

```yaml
apiVersion: v1
kind: Secret
metadata: { name: db-credentials }
type: Opaque
data:
  username: cG9zdGdyZXM=          # "postgres" base64
  password: c3VwM3JfczNjcjN0      # base64
stringData:
  api_token: "raw-string-token"   # API server base64-encodes it on write
```

What's different from a ConfigMap:

- **Storage** — YAML uses `data:` (base64) or `stringData:` (raw, encoded on write).
- **In-memory mounting** — a mounted Secret lands on `tmpfs` (RAM), never a node's disk.
- **etcd storage** — by default Secret data sits in `etcd` as base64, **not encrypted.** Production enables an `EncryptionConfiguration` so the **API server** (not etcd) encrypts it. The CKA wants: *optional but recommended.*
- **RBAC** — treat Secrets as more sensitive by policy.

> **base64 is encoding, not encryption.** Anyone with `get secret` decodes it instantly: `echo cG9zdGdyZXM= | base64 -d` → `postgres`. Committing a Secret YAML to Git commits plaintext.

Create them from literals, files, or typed shortcuts:

```bash
kubectl create secret generic db-credentials \
  --from-literal=username=postgres --from-literal=password='sup3r_s3cr3t'
kubectl create secret tls web-tls --cert=server.crt --key=server.key
kubectl create secret docker-registry regcred --docker-server=... --docker-username=... --docker-password=...
```

(If `data` and `stringData` both set a key, `stringData` wins.) Production-grade secret management — Sealed Secrets, External Secrets Operator, SOPS — is standard but out of CKA scope. On our map this is the **Secret** chip in the Config box: same shape as ConfigMap, tighter handling.